## 5. GRU Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [2]:
import json

from src.dataloader import load_cgm_stats, load_tensors
from src import metrics
from src import train
from src.models.nn_baseline import GRUPredictor, HORIZONS

MODEL_NAME = 'GRU'
SEED = 42  # change this and re-run for a multi-seed stability comparison
# Built from REPO_ROOT, not the current working directory: Colab's cwd is /content regardless of
# where this notebook file itself lives on the mounted Drive, so a plain relative path like
# Path('results') would silently write to the wrong (ephemeral) place.
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_PATH = EXPERIMENT_DIR / 'results' / f'gru_seed{SEED}_results.json'
CHECKPOINT_DIR = EXPERIMENT_DIR / 'checkpoints' / f'gru_seed{SEED}'

In [4]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

print('n_train_batches', len(train_loader), 'n_val_batches', len(val_loader),
      'n_test_batches', len(test_loader))

sample_cgm, sample_target = next(iter(train_loader))
print('input batch shape (cgm):', tuple(sample_cgm.shape), sample_cgm.dtype)
print('target batch shape:', tuple(sample_target.shape), sample_target.dtype)

n_train_batches 327 n_val_batches 66 n_test_batches 66
input batch shape (cgm): (256, 24) torch.float32
target batch shape: (256, 5) torch.float32


In [5]:
train.set_seed(SEED)  # before constructing the model -- weight init draws from this
model = GRUPredictor()

model, history = train.train_model(
    model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
    checkpoint_dir=CHECKPOINT_DIR,
)

device: cpu  model: GRUPredictor  params: 157,957
epoch   1/50  train_loss=0.4547  val_loss=0.3566  val_rmse(mg/dL)=[11.8, 18.3, 25.9, 30.4, 33.2]
epoch   2/50  train_loss=0.4021  val_loss=0.3434  val_rmse(mg/dL)=[11.1, 17.7, 25.4, 30.0, 32.7]
epoch   3/50  train_loss=0.3918  val_loss=0.3465  val_rmse(mg/dL)=[11.4, 18.0, 25.5, 30.0, 32.7]
epoch   4/50  train_loss=0.3890  val_loss=0.3427  val_rmse(mg/dL)=[11.4, 17.9, 25.4, 29.9, 32.5]
epoch   5/50  train_loss=0.3877  val_loss=0.3392  val_rmse(mg/dL)=[11.2, 17.8, 25.3, 29.7, 32.4]
epoch   6/50  train_loss=0.3853  val_loss=0.3336  val_rmse(mg/dL)=[11.1, 17.7, 25.1, 29.4, 32.1]
epoch   7/50  train_loss=0.3844  val_loss=0.3403  val_rmse(mg/dL)=[11.1, 17.8, 25.3, 29.8, 32.4]
epoch   8/50  train_loss=0.3835  val_loss=0.3416  val_rmse(mg/dL)=[11.3, 17.9, 25.4, 29.8, 32.4]
epoch   9/50  train_loss=0.3822  val_loss=0.3372  val_rmse(mg/dL)=[11.1, 17.7, 25.2, 29.6, 32.3]
epoch  10/50  train_loss=0.3823  val_loss=0.3360  val_rmse(mg/dL)=[11.2, 17.8

In [ ]:
y_pred_val, y_true_val = train.predict(model, val_loader)
y_pred_test, y_true_test = train.predict(model, test_loader)

val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

print('=== Validation (best checkpoint) ===')
print(metrics.format_results_table(val_results))
print()
print('=== Test ===')
print(metrics.format_results_table(test_results))

In [6]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seed': SEED,
        'val_results': val_results,
        'test_results': test_results,
        'n_epochs_trained': len(history['train_loss']),
    }, f, indent=2)

print(f'saved to {RESULTS_PATH}')
print(f'checkpoints (best.pt / final.pt / history.json) saved to {CHECKPOINT_DIR}')

NameError: name 'val_results' is not defined